# 🕸️ Five Multi-Agent Architectures in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Network architecture** - every agent can hand off to every other agent, with no central coordinator
2. **Supervisor architecture** - one node makes all routing decisions and workers always return to it
3. **Supervisor with tool-calling** - agents exposed as *tools*, letting the model's own tool-calling do the routing
4. **Hierarchical architecture** - compiled team subgraphs nested under a top-level coordinator
5. **Custom workflow architecture** - explicit, mostly-fixed paths with a branch point in the middle
6. **`Command(goto=...)`** - the primitive underneath all five, which lets a node choose its own successor

## Prerequisites
- Core LangGraph concepts: `StateGraph`, `START`/`END`, nodes and edges — see [repository 04 — LangGraph fundamentals](https://github.com/Sourav692/ai-engineering-demystified-04-langgraph-fundamentals)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (every worker calls the model; the ping/pong cap demo does not)

## Where this fits
`02_multiagent_pattern_from_scratch.ipynb` wired agents into a fixed DAG. This notebook is the map of the
territory: the only one that builds all five LangGraph architectures side by side. Modules 06–09 are deep
dives into them.

## 📐 The five architectures at a glance

| # | Architecture | Who decides what runs next | Typical use |
|---|---|---|---|
| 1 | **Network** | Each agent decides for itself | Open-ended collaboration, peer review |
| 2 | **Supervisor** | One central node | Clear division of labour, auditable routing |
| 3 | **Supervisor w/ tool-calling** | The model, via tool selection | Intent routing, support desks |
| 4 | **Hierarchical** | A coordinator over team coordinators | Org-shaped problems, many specialists |
| 5 | **Custom workflow** | Mostly the graph; one dynamic branch | Pipelines with a decision point |

**Origin:** LangGraph's multi-agent guides; conceptual ancestors are AutoGen (Microsoft, 2023) and CrewAI (2023).

### Key Concepts:
- **`Command(goto=..., update=...)`**: a node's return value that does two jobs at once — write to state *and* name the next node. This is what makes non-linear control flow possible without pre-wiring every edge.
- **`MessagesState`**: LangGraph's built-in state with a single `messages` channel whose reducer appends. Because it appends, nodes return only their *new* messages — never the whole list.
- **Static edges vs. dynamic routing**: `add_edge` fixes a path at build time; `Command(goto=)` picks one at run time. Most real systems use both.

---
## 📦 Step 1: Environment Setup

One setup cell for the whole notebook: imports, the recursion cap every run uses, the LLM every worker calls,
and `show_graph()`, which prints each graph's nodes and edges as text.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, Credentials, LLM, and Graph Printing
# ============================================================================
import warnings
from typing import Annotated, Literal

from aied_helpers import get_openai_llm
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import InjectedState, tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))

# --- Safety cap -----------------------------------------------------------
# LangGraph counts SUPER-STEPS, not "iterations", and the default cap is
# 10,007 - high enough that a graph which never reaches END will churn for a
# very long time before failing. Every invoke below passes this limit
# explicitly so a routing bug surfaces in seconds instead of minutes.
RECURSION_LIMIT = 25

llm = get_openai_llm(model="gpt-4o-mini")


def agent_reply(role: str, instruction: str, state: MessagesState) -> AIMessage:
    """One model call. The role tag is added here so routers never parse model prose."""
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    f"{instruction}\n\n"
                    "Reply in at most 3 sentences. Do not name the next agent."
                )
            ),
            *state.get("messages", []),
        ]
    )
    return AIMessage(content=f"[{role}]: {response.content}")


def show_graph(compiled_graph):
    """Print a compiled graph's nodes and edges (dashed/conditional edges are marked)."""
    graph = compiled_graph.get_graph()
    print("Nodes:", [n for n in graph.nodes if n not in ("__start__", "__end__")])
    for edge in graph.edges:
        print(f"  {edge.source} -> {edge.target}{'  (dynamic)' if edge.conditional else ''}")


print(f"✅ Setup complete — LLM {llm.model_name}, show_graph(), recursion cap = {RECURSION_LIMIT} super-steps")

---
## 🕸️ Part 1: Network Architecture

There is no coordinator. Each node inspects state and names its own successor with `Command(goto=...)`.
Only `START → researcher` is fixed at build time.

In this example the successor sets are small, and one of them points backward: the reviewer hands the draft
back to the writer once, then ends the run on the second visit.

> **Note**: The `Literal[...]` in each return annotation is not decoration. It declares the set of
> nodes this one may jump to, and LangGraph uses it to infer the edges `show_graph()` prints.

### How this network is wired

A network has no supervisor node. Each agent reads the shared messages, writes its own reply, and returns `Command(goto=...)` to name the next agent. `agent_reply` adds the role tag (`[RESEARCHER]`, `[WRITER]`, `[REVIEWER]`). The `if` matches that tag, so the route stays put when the model's prose changes.

`Literal[...]` is the edge list LangGraph draws in `show_graph()`. Every name in it must be a branch the function can take. A name that the body never assigns still appears as an edge.

The previous snippet looked like a mesh and ran as a straight line. The user message has no `[researcher]` tag, so the researcher went to the writer and then appended `[RESEARCHER]`. The writer saw that tag, went to the reviewer, and appended `[WRITER]`. The reviewer saw that tag and went to `END`. The `else` branches asked for a transcript that this order can never produce.

![Previous run. START, researcher, writer, reviewer, END.](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/05_multi_agent_foundations/notebooks/figures/network_previous_run.svg)

Solid arrows in the next diagram are that run. Dotted arrows were in the type, or in an else, and the graph never took them.

![Previous edges. Solid arrows ran. Dotted arrows were unreachable.](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/05_multi_agent_foundations/notebooks/figures/network_previous_edges.svg)

The cell below keeps a Literal target only when the run can take it. The researcher always hands off to the writer. The writer always hands off to the reviewer. The reviewer sends the draft back to the writer on the first visit, because no reviewer tag exists yet. On the second visit that tag is already in the transcript, so the reviewer goes to END.

![Current run. The reviewer returns to the writer once, then ends.](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/05_multi_agent_foundations/notebooks/figures/network_current_run.svg)

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Agent Definitions
# ============================================================================
# Each agent reads shared state, produces its own contribution, and names the
# next agent to run. Nobody coordinates them.


def researcher_agent(state: MessagesState) -> Command[Literal["writer_agent"]]:
    """Research a topic, then hand the findings to the writer."""
    reply = agent_reply(
        "RESEARCHER",
        "You research multi-agent systems. Give three concrete findings the writer can use.",
        state,
    )
    return Command(goto="writer_agent", update={"messages": [reply]})


def writer_agent(state: MessagesState) -> Command[Literal["reviewer_agent"]]:
    """Draft or revise, then hand the text to the reviewer."""
    messages = state.get("messages", [])
    revising = any("[reviewer]" in str(msg).lower() for msg in messages)
    instruction = (
        "A review is already in the conversation. Revise the draft using that review."
        if revising
        else "Draft a short article from the research already in the conversation."
    )
    reply = agent_reply("WRITER", instruction, state)
    return Command(goto="reviewer_agent", update={"messages": [reply]})


def reviewer_agent(state: MessagesState) -> Command[Literal["writer_agent", END]]:
    """Review once, send the draft back, then approve on the second visit."""
    messages = state.get("messages", [])
    already_reviewed = any("[reviewer]" in str(msg).lower() for msg in messages)
    instruction = (
        "You already reviewed this draft. Approve the revision in one sentence."
        if already_reviewed
        else "Review the draft. Name one strength and one fix."
    )
    reply = agent_reply("REVIEWER", instruction, state)
    if already_reviewed:
        next_agent = END
    else:
        next_agent = "writer_agent"
    return Command(goto=next_agent, update={"messages": [reply]})


print("✅ Network agents defined: researcher, writer, reviewer")

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Build and Compile
# ============================================================================
# Only ONE static edge exists — START to the researcher. Every other transition
# is chosen at runtime by a Command(goto=...).

builder = StateGraph(MessagesState)
builder.add_node("researcher_agent", researcher_agent)
builder.add_node("writer_agent", writer_agent)
builder.add_node("reviewer_agent", reviewer_agent)

builder.add_edge(START, "researcher_agent")
network = builder.compile()

print("✅ Network compiled — 3 nodes, 1 static edge, the rest dynamic")

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Visualize
# ============================================================================
# The edges marked (dynamic) are the ones LangGraph inferred from the Literal annotations.

show_graph(network)

In [ ]:
network

In [ ]:
# ============================================================================
# NETWORK ARCHITECTURE: Run
# ============================================================================

result = network.invoke(
    {"messages": ["Write a short brief on how a network of agents collaborates."]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🔧 Message trace:")
for message in result["messages"]:
    print(f"   {message}")

### What to notice

The trace has five agent messages: researcher, writer, reviewer, writer, reviewer. The hop from the
reviewer back to the writer is the network. No `add_edge` wires it. The reviewer ends the run on its
second visit because a `[REVIEWER]` tag is already in the transcript. The prose changes between runs.
That tag check does not.

**The trade-off:** flexibility with no termination guarantee. Nothing in a network stops two agents
bouncing work between themselves forever; only the agents' own conditions end the run. Compare that
against the supervisor in Part 2, where exactly one node decides when work is finished.

#### 🛡️ Seeing the safety cap work

The paragraph above is not hypothetical, so here is the failure and the guard against it.

Two agents below hand work to each other unconditionally — a routing bug you could plausibly write
by accident. Without a cap this runs **10,007 super-steps** before LangGraph gives up. With
`recursion_limit` it stops in a fraction of a second and tells you what happened.

> **Note**: this caps *steps*, not wall-clock time. Twenty-five steps of a graph that calls an LLM
> at every node is still slow — it is a runaway-loop guard, not a timeout.

In [ ]:
# ============================================================================
# SAFETY CAP: A Deliberately Non-Terminating Network
# ============================================================================
# ping and pong route to each other forever and never reach END.


def ping(state: MessagesState) -> Command[Literal["pong"]]:
    """Always hands off to pong. Never ends."""
    return Command(goto="pong", update={"messages": ["[PING]"]})


def pong(state: MessagesState) -> Command[Literal["ping"]]:
    """Always hands off to ping. Never ends."""
    return Command(goto="ping", update={"messages": ["[PONG]"]})


loop_builder = StateGraph(MessagesState)
loop_builder.add_node("ping", ping)
loop_builder.add_node("pong", pong)
loop_builder.add_edge(START, "ping")
runaway = loop_builder.compile()

try:
    runaway.invoke({"messages": []}, config={"recursion_limit": RECURSION_LIMIT})
    print("❌ Unexpected: the runaway graph terminated on its own")
except GraphRecursionError:
    print(f"✅ Caught GraphRecursionError after {RECURSION_LIMIT} super-steps")
    print("   Without the cap this would have run 10,007 steps before failing.")
    print("   Catch it explicitly when a partial result is still useful; otherwise")
    print("   let it raise, because a graph that cannot stop is a bug, not a slow run.")

---
## 🧑‍💼 Part 2: Supervisor Architecture

A single `supervisor` node makes every routing decision, and workers always return to it. Control
flow becomes a star: supervisor → worker → supervisor → worker → … → `END`.

This is the most common production shape, because the routing logic lives in exactly one place you
can read, log and test.

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Supervisor and Worker Nodes
# ============================================================================
# Note the asymmetry in the return types: the supervisor may go to any worker
# or END, but each worker may only go back to the supervisor.


def supervisor(state: MessagesState) -> Command[Literal["content_creator", "editor", END]]:
    """Decide which worker runs next, or end the run. This node does not call the model."""
    messages = state.get("messages", [])
    last_message = str(messages[-1]).lower() if messages else ""

    if "[editor]" in last_message:
        return Command(goto=END)
    if "[creator]" in last_message:
        return Command(goto="editor")
    return Command(goto="content_creator")


def content_creator(state: MessagesState) -> Command[Literal["supervisor"]]:
    """Produce the first draft, then report back."""
    reply = agent_reply(
        "CREATOR",
        "You write a short introduction to multi-agent systems for a technical reader.",
        state,
    )
    return Command(goto="supervisor", update={"messages": [reply]})


def editor(state: MessagesState) -> Command[Literal["supervisor"]]:
    """Polish the draft, then report back."""
    reply = agent_reply(
        "EDITOR",
        "You edit the creator's draft. Return the polished version only.",
        state,
    )
    return Command(goto="supervisor", update={"messages": [reply]})


print("✅ Supervisor and workers defined: supervisor, content_creator, editor")

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Build and Compile
# ============================================================================

builder = StateGraph(MessagesState)
builder.add_node("supervisor", supervisor)
builder.add_node("content_creator", content_creator)
builder.add_node("editor", editor)

builder.add_edge(START, "supervisor")
supervisor_system = builder.compile()

print("✅ Supervisor system compiled")

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Visualize
# ============================================================================

show_graph(supervisor_system)

In [ ]:
# ============================================================================
# SUPERVISOR ARCHITECTURE: Run
# ============================================================================
# The user message has no worker tag, so the supervisor's first decision is the creator.

result = supervisor_system.invoke(
    {"messages": ["Introduce multi-agent systems in a short paragraph."]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🔧 Message trace:")
for message in result["messages"]:
    print(f"   {message}")

### What to notice

Only the workers' output appears in the trace — the supervisor routes without writing to state.
That is deliberate and worth copying: a router that does not add messages keeps the transcript
readable.

**The trade-off:** every decision funnels through one node, which is both the strength (one place to
audit) and the weakness (a single point of failure, and a bottleneck when workers could have run in
parallel).

---
## 🛠️ Part 3: Supervisor with Tool-Calling

Instead of writing routing logic by hand, expose each agent as a **tool** and let the model's own
tool-calling choose. The supervisor becomes a standard agent; "routing" is just tool selection.

`InjectedState` (from `langchain.tools`) is the piece that makes these agents rather than plain
functions: the graph's state is injected into the tool call without the model having to produce it as an
argument. Each tool below reports how many messages of the conversation it received.
`ToolRuntime`, also in `langchain.tools`, is the broader handle — state plus context, store and stream writer —
that module 03 uses for its memory tools.

> **API note:** the agent comes from `langchain.agents.create_agent`, which replaces the deprecated
> `langgraph.prebuilt.create_react_agent`. Its prompt argument is `system_prompt` (it was `prompt`).

In [ ]:
# ============================================================================
# TOOL-CALLING SUPERVISOR: Agents Exposed as Tools
# ============================================================================
# The docstring is not documentation here — it is the tool description the model
# reads to decide which specialist to call. Write it for the model.


def _specialist_reply(role: str, instruction: str, query: str) -> str:
    """One model call inside a tool. The supervisor already chose this specialist."""
    response = llm.invoke(
        [
            SystemMessage(content=f"You are the {role}. {instruction} Reply in at most 2 sentences."),
            HumanMessage(content=query),
        ]
    )
    return response.content


@tool
def billing_agent(query: str, state: Annotated[dict, InjectedState]) -> str:
    """Handle billing-related queries and issues."""
    answer = _specialist_reply(
        "billing agent",
        "Resolve the billing issue. Mention an account credit when a charge is disputed.",
        query,
    )
    return f"Billing Agent ({len(state['messages'])} messages of context): {answer}"


@tool
def technical_agent(query: str, state: Annotated[dict, InjectedState]) -> str:
    """Handle technical support and troubleshooting."""
    answer = _specialist_reply(
        "technical agent",
        "Give a short troubleshooting step for the reported failure.",
        query,
    )
    return f"Technical Agent ({len(state['messages'])} messages of context): {answer}"


@tool
def general_agent(query: str, state: Annotated[dict, InjectedState]) -> str:
    """Handle general inquiries and information requests."""
    answer = _specialist_reply(
        "general agent",
        "Answer the inquiry directly.",
        query,
    )
    return f"General Agent ({len(state['messages'])} messages of context): {answer}"


agent_tools = [billing_agent, technical_agent, general_agent]

print("✅ Agent-tools defined:", [t.name for t in agent_tools])

In [ ]:
# ============================================================================
# TOOL-CALLING SUPERVISOR: Build
# ============================================================================
# No StateGraph and no routing code — create_agent supplies the loop, and the
# model's tool selection IS the routing decision.

tool_supervisor = create_agent(llm, tools=agent_tools)

print("✅ Tool-calling supervisor ready")

In [ ]:
# ============================================================================
# TOOL-CALLING SUPERVISOR: Run
# ============================================================================
# A deliberately mixed query — billing AND technical — to see whether the model
# routes to more than one specialist.

result = tool_supervisor.invoke(
    {"messages": [("user", "I'm having trouble with my bill and the app keeps crashing")]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🤖 Conversation trace:")
for message in result["messages"]:
    kind = type(message).__name__
    content = getattr(message, "content", str(message))
    calls = [f"{tc['name']}({tc['args']})" for tc in getattr(message, "tool_calls", []) or []]
    print(f"calls {calls}")
    print(f"   {kind}: {content or '(no text)'}")
    if calls:
        print(f"      ↳ tool calls: {calls}")  # the model sends only `query`; `state` is injected

### What to notice

Check which specialists actually got called. A mixed query *may* produce two tool calls, or the
model may decide one covers it — that variability is the point of this architecture. Routing quality
is now a prompt-and-description problem, not a code problem.

**The trade-off:** you gain flexibility and lose determinism. In Part 2 you could read the routing
rules; here you can only influence them through tool descriptions, and the same input can route
differently between runs.

---
## 🏢 Part 4: Hierarchical Architecture

Two levels of supervision. Each **team** is its own `StateGraph`, compiled independently, then added
to the parent graph as a single node — a compiled graph is a valid node because it has the same
invoke interface as a function.

The structure below is an org chart: a top coordinator over a dev team and a QA team, each with
their own supervisor and their own specialists.

### Key Concepts:
- **Subgraph as a node**: `builder.add_node("dev_team", dev_team)` where `dev_team` is a *compiled graph*. The parent never sees inside it.
- **Shared state contract**: this works only because every level uses `MessagesState`. If teams used different schemas you would need adapter code at each boundary.
- **Test teams in isolation first**: each team compiles and runs on its own, which is how you debug a hierarchy without running the whole thing.

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Development Team Subgraph
# ============================================================================
# A complete, self-contained graph — its own supervisor and its own workers.


def dev_supervisor(state: MessagesState) -> Command[Literal["frontend_dev", "backend_dev", END]]:
    """Route work within the development team until both specialists are done."""
    messages = state.get("messages", [])
    all_messages = " ".join(str(msg).lower() for msg in messages)

    # Tags come from agent_reply, so completion does not depend on the model saying "complete".
    backend_done = "[backend]" in all_messages
    frontend_done = "[frontend]" in all_messages

    if not backend_done:
        return Command(goto="backend_dev")
    elif not frontend_done:
        return Command(goto="frontend_dev")
    else:
        return Command(goto=END)


def backend_dev(state: MessagesState) -> Command[Literal["dev_supervisor"]]:
    """Backend specialist."""
    reply = agent_reply(
        "BACKEND",
        "You are the backend developer. Name the API and the data store for this project.",
        state,
    )
    return Command(goto="dev_supervisor", update={"messages": [reply]})


def frontend_dev(state: MessagesState) -> Command[Literal["dev_supervisor"]]:
    """Frontend specialist."""
    reply = agent_reply(
        "FRONTEND",
        "You are the frontend developer. Name the screens you will build for this project.",
        state,
    )
    return Command(goto="dev_supervisor", update={"messages": [reply]})


dev_builder = StateGraph(MessagesState)
dev_builder.add_node("dev_supervisor", dev_supervisor)
dev_builder.add_node("backend_dev", backend_dev)
dev_builder.add_node("frontend_dev", frontend_dev)
dev_builder.add_edge(START, "dev_supervisor")
dev_team = dev_builder.compile()

print("✅ Dev team compiled — it is a standalone graph, runnable on its own")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: QA Team Subgraph
# ============================================================================
# Same shape as the dev team. Building both the same way is what lets the parent
# treat them interchangeably.


def qa_supervisor(state: MessagesState) -> Command[Literal["tester", "security_analyst", END]]:
    """Route work within the QA team until testing and security are both done."""
    messages = state.get("messages", [])
    all_messages = " ".join(str(msg).lower() for msg in messages)

    testing_done = "[tester]" in all_messages
    security_done = "[security]" in all_messages

    if not testing_done:
        return Command(goto="tester")
    elif not security_done:
        return Command(goto="security_analyst")
    else:
        return Command(goto=END)


def tester(state: MessagesState) -> Command[Literal["qa_supervisor"]]:
    """Functional testing specialist."""
    reply = agent_reply(
        "TESTER",
        "You are QA. Name two tests for the project described in the conversation.",
        state,
    )
    return Command(goto="qa_supervisor", update={"messages": [reply]})


def security_analyst(state: MessagesState) -> Command[Literal["qa_supervisor"]]:
    """Security review specialist."""
    reply = agent_reply(
        "SECURITY",
        "You are security. Name one risk and one control for this project.",
        state,
    )
    return Command(goto="qa_supervisor", update={"messages": [reply]})


qa_builder = StateGraph(MessagesState)
qa_builder.add_node("qa_supervisor", qa_supervisor)
qa_builder.add_node("tester", tester)
qa_builder.add_node("security_analyst", security_analyst)
qa_builder.add_edge(START, "qa_supervisor")
qa_team = qa_builder.compile()

print("✅ QA team compiled")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Test Each Team in Isolation
# ============================================================================
# Do this BEFORE assembling the hierarchy. A team that misbehaves here will be
# far harder to diagnose once it is one node inside a larger graph.

for name, team in [("dev_team", dev_team), ("qa_team", qa_team)]:
    outcome = team.invoke(
        {"messages": [f"Kick off {name} for a small note-taking app."]},
        config={"recursion_limit": RECURSION_LIMIT},
    )
    print(f"🔧 {name}: {len(outcome['messages'])} messages")
    for message in outcome["messages"]:
        print(f"      {message}")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Top-Level Coordinator and Assembly
# ============================================================================
# The compiled teams are added as ordinary nodes. Static edges return control to
# the top supervisor after each team finishes.


def top_supervisor(state: MessagesState) -> Command[Literal["dev_team", "qa_team", END]]:
    """Coordinate the two teams: development first, then QA."""
    messages = state.get("messages", [])

    if not messages:
        return Command(goto="dev_team")

    all_messages = " ".join(str(msg).lower() for msg in messages)

    dev_complete = "[backend]" in all_messages and "[frontend]" in all_messages
    qa_complete = "[tester]" in all_messages and "[security]" in all_messages

    if not dev_complete:
        return Command(goto="dev_team")
    elif not qa_complete:
        return Command(goto="qa_team")
    else:
        return Command(goto=END)


builder = StateGraph(MessagesState)
builder.add_node("top_supervisor", top_supervisor)
builder.add_node("dev_team", dev_team)  # a COMPILED GRAPH used as a node
builder.add_node("qa_team", qa_team)

builder.add_edge(START, "top_supervisor")
builder.add_edge("dev_team", "top_supervisor")
builder.add_edge("qa_team", "top_supervisor")

hierarchical_system = builder.compile()

print("✅ Hierarchy compiled — 2 levels, 3 supervisors, 4 specialists")

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Visualize
# ============================================================================
# The parent graph shows three nodes. The teams' internals are deliberately
# hidden — that encapsulation is the whole benefit of subgraphs.

show_graph(hierarchical_system)

In [ ]:
# ============================================================================
# HIERARCHICAL ARCHITECTURE: Run
# ============================================================================
result = hierarchical_system.invoke(
    {"messages": ["Start a small note-taking app."]},
    # A hierarchy takes many more super-steps than a flat graph: each team run
    # is several steps of its own, so the flat cap is doubled here.
    config={"recursion_limit": RECURSION_LIMIT * 2},
)

print("🔧 Full trace across both levels:")
for i, message in enumerate(result["messages"], 1):
    print(f"   {i}. {message}")

### What to notice

Messages from both teams land in one flat transcript, because parent and subgraphs share the same
`MessagesState`. That shared contract is what lets a compiled graph drop in as a node with no
adapter code — and it is also why the top supervisor can inspect team output directly.

**The trade-off:** encapsulation costs you visibility. The parent cannot see *why* a team decided
anything, only what it appended. Raising `recursion_limit` is the tell: hierarchies take many more
steps, and step budget becomes something you have to think about.

---
## 🔀 Part 5: Custom Workflow Architecture

Mostly a fixed pipeline, with one genuine decision point. `document_parser → content_analyzer →
(text_processor | image_processor) → summarizer`.

This is the architecture most real systems actually need: the sequence is known, and only one or two
places require a judgement call.

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Pipeline Stages
# ============================================================================
# Every stage but one has a single possible successor. content_analyzer is the
# only real branch — note it is the only Literal with two options.


class ContentKind(BaseModel):
    """The one judgement in this pipeline: which processor should run."""

    kind: Literal["text", "image"] = Field(
        description="Use image only when the document has no prose. Otherwise use text."
    )
    note: str = Field(description="One sentence on what the document contains.")


analyzer_llm = llm.with_structured_output(ContentKind)


def document_parser(state: MessagesState) -> Command[Literal["content_analyzer"]]:
    """Stage 1 — parse the incoming document."""
    reply = agent_reply(
        "PARSER",
        "You parse the uploaded document. State its format and a one-line outline.",
        state,
    )
    return Command(goto="content_analyzer", update={"messages": [reply]})


def content_analyzer(state: MessagesState) -> Command[Literal["text_processor", "image_processor"]]:
    """Stage 2 — the branch point. The model classifies; the graph only follows."""
    decision = analyzer_llm.invoke(
        [
            SystemMessage(
                content=(
                    "Classify the parsed document. Choose image only when it has no prose to extract. "
                    "Otherwise choose text."
                )
            ),
            *state["messages"],
        ]
    )
    goto = "text_processor" if decision.kind == "text" else "image_processor"
    return Command(
        goto=goto,
        update={"messages": [AIMessage(content=f"[ANALYZER]: {decision.note}")]},
    )


def text_processor(state: MessagesState) -> Command[Literal["summarizer"]]:
    """Stage 3a — extract and clean text."""
    reply = agent_reply(
        "TEXT_PROC",
        "Extract the prose from the document and list its main claims.",
        state,
    )
    return Command(goto="summarizer", update={"messages": [reply]})


def image_processor(state: MessagesState) -> Command[Literal["summarizer"]]:
    """Stage 3b — analyse figures and diagrams."""
    reply = agent_reply(
        "IMAGE_PROC",
        "Describe the figures in the document. If none are present, say so.",
        state,
    )
    return Command(goto="summarizer", update={"messages": [reply]})


def summarizer(state: MessagesState) -> Command[Literal[END]]:
    """Stage 4 — converge both branches into one summary."""
    reply = agent_reply(
        "SUMMARIZER",
        "Write a two-sentence summary of the processed document.",
        state,
    )
    return Command(goto=END, update={"messages": [reply]})


print("✅ Workflow stages defined: parser → analyzer → (text | image) → summarizer")

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Build and Compile
# ============================================================================

builder = StateGraph(MessagesState)
builder.add_node("document_parser", document_parser)
builder.add_node("content_analyzer", content_analyzer)
builder.add_node("text_processor", text_processor)
builder.add_node("image_processor", image_processor)
builder.add_node("summarizer", summarizer)

builder.add_edge(START, "document_parser")
custom_workflow = builder.compile()

print("✅ Custom workflow compiled — 5 stages, 1 branch point")

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Visualize
# ============================================================================
# This is the only edge list of the five that reads like a flowchart, because it
# is the only architecture that mostly IS one.

show_graph(custom_workflow)

In [ ]:
# ============================================================================
# CUSTOM WORKFLOW: Run
# ============================================================================

result = custom_workflow.invoke(
    {"messages": [(
        "Process this document:\n\n"
        "Title: Multi-agent systems\n"
        "Body: A supervisor routes work. A network lets peers hand off. A hierarchy nests teams."
    )]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print("🔧 Pipeline trace:")
for message in result["messages"]:
    print(f"   {message}")

### What to notice

Only one branch ran. `content_analyzer` called the model, classified this sample as text, and
`image_processor` never executed. Both branches converge on `summarizer`, so the graph has a diamond
shape rather than a fork that never rejoins. Re-run with a figures-only document and the other branch fires.

**The trade-off:** predictability at the cost of adaptability. This graph does the same thing every
time apart from one decision, which is exactly what you want for a document pipeline and exactly
what you do not want for open-ended research.

---
## 📝 Summary

### 1. One primitive underneath all five
- **`Command(goto=..., update=...)`** writes to state and names the next node in a single return. Every architecture here is a different discipline imposed on that one capability.
- **`Literal[...]` return annotations** declare a node's legal successors, and LangGraph reads them to build the graph's edges.

### 2. The five architectures, compared

| Architecture | Control | Fails when | Reach for it when |
|---|---|---|---|
| **Network** | Distributed | Nothing guarantees termination | Agents genuinely need to negotiate |
| **Supervisor** | Centralised | The supervisor is a bottleneck | Routing must be auditable |
| **Tool-calling supervisor** | Model-driven | Routing is non-deterministic | Intent classification is the hard part |
| **Hierarchical** | Layered | Step budgets grow; internals are opaque | The problem is org-shaped |
| **Custom workflow** | Graph-driven | Too rigid for open-ended work | The sequence is known in advance |

### 3. Design lessons worth carrying forward
- **A router that writes nothing keeps transcripts readable** — the Part 2 supervisor never appends a message.
- **Test subgraphs in isolation before assembling** — Part 4 runs each team alone first, which is the only practical way to debug a hierarchy.
- **One shared state schema removes adapter code** — the hierarchy composes only because every level speaks `MessagesState`.
- **Always pass `recursion_limit`** — the default is 10,007 super-steps, so an unguarded routing bug burns minutes before it fails. It is the graph-level counterpart of the old `AgentExecutor(max_iterations=…)`; for an agent built with `create_agent`, `ModelCallLimitMiddleware` and `ToolCallLimitMiddleware` (from `langchain.agents.middleware`) cap model and tool calls directly.
- **Raise `recursion_limit` deliberately, not reflexively** — needing it is a signal about how many steps your design really takes.

### Next Steps
- `04_agent_communication.ipynb` (this module) — how agents pass information: messages, typed state, and a generator–critic loop over shared state.
- `01_supervisor_pattern.ipynb` (module 06) — the supervisor pattern built slowly, with a Pydantic `RouteDecision` replacing the role-tag checks used here.
- `02_hierarchical_agents.ipynb` (module 06) — hierarchy done properly, with a purpose-built `TeamState` instead of `MessagesState`.
- `02_multi_agent_swarm.ipynb` (module 07) — the network idea taken to production, with real handoff tools and persistent memory.

> **A note on the demos:** every worker calls the model for its message. Routing in Parts 1, 2, and 4
> still keys off the role tag this notebook adds (`[RESEARCHER]`, `[CREATOR]`, `[BACKEND]`, …), not off
> the model's wording, so the topology stays readable when the prose changes. Part 3 lets the model
> choose tools, and each tool calls the model too. Part 5 lets the model choose the one branch.
> The ping/pong cell stays free of model calls so the recursion cap fails in a fraction of a second.